Import Libraries

In [4]:
import random
import string

from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, END

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

from dotenv import load_dotenv

load_dotenv()
print("Imports are Successful!!!")

Imports are Successful!!!


Define the State

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list, add]

print("State defined!!!")

State defined!!!


Initialize Claude

In [8]:
llm = ChatAnthropic(
    model = "claude-sonnet-4-6",
    temperature = 0
)

print("Claude is Ready!!!")

Claude is Ready!!!


Define Math Tools

In [21]:
import math

@tool
def calculator(expression: str) -> str:
    """Evaluates a basic math expression like '2 + 2' or '15 * 4'."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def square_root(number: float) -> str:
    """Calculates the square root of a number."""
    try:
        result = math.sqrt(number)
        return f"The square root of {number} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def power(base: float, exponent: float) -> str:
    """Raises base to the power of exponent."""
    try:
        result = math.pow(base, exponent)
        return f"{base} to the power of {exponent} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator, square_root, power]
llm_with_tools = llm.bind_tools(tools)
tool_node_map = {t.name: t for t in tools}
print("Tools bound to Claude!!!")

Tools bound to Claude!!!


LangGraph Agent

In [22]:
# Agent node - takes state, calls Claude, returns updated messages
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke([SystemMessage(content= """You are a helpful math assistant.
    You have access to the calculator, square_root and power tools.
    Always use the appropriate tool to solve math probalems accurately.""")] + state["messages"]
                                    )
    return {"messages": [response]}

# Tool node - takes state, finds last tool call, executes it, returns result
def tool_node(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    results = []
    for tool_call in last_message.tool_calls:
        tool = tool_node_map[tool_call["name"]]
        result = tool.invoke(tool_call["args"])
        results.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )
    return {"messages": results}
print("Langgraph Agent Ready!!!")

Langgraph Agent Ready!!!


Conditional Edge Function

In [23]:
def should_use_tool(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return "end"

Build the Graph

In [24]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)
workflow.add_node("agent_node",agent_node)
workflow.add_node("tool_node", tool_node)
workflow.add_edge("tool_node", "agent_node")
workflow.add_conditional_edges("agent_node", should_use_tool, {
    "tool_node": "tool_node",
    "end": END,
})
workflow.set_entry_point("agent_node")

In [25]:
app = workflow.compile()
result = app.invoke({"messages": [HumanMessage(content="What is 10 to the power of 3")]})

print("\n Final Answer:")
print(result["messages"][-1].content)


 Final Answer:
**10 to the power of 3 is 1,000!** 🎉

This means 10 × 10 × 10 = **1,000**. It's also commonly known as one thousand, or 10³ in exponential notation.


Inspect Full State

In [26]:
print("Full message flow:")
print("-" * 50)
for msg in result["messages"]:
    if isinstance(msg, HumanMessage):
        print(f"👤 Human: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f"🤖 Agent decided to use: {[tc['name'] for tc in msg.tool_calls]}")
        else:
            print(f"🤖 Agent: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"🔧 Tool result: {msg.content}")
print("-" * 50)

Full message flow:
--------------------------------------------------
👤 Human: What is 10 to the power of 3
🤖 Agent decided to use: ['power']
🔧 Tool result: 10.0 to the power of 3.0 is 1000.0
🤖 Agent: **10 to the power of 3 is 1,000!** 🎉

This means 10 × 10 × 10 = **1,000**. It's also commonly known as one thousand, or 10³ in exponential notation.
--------------------------------------------------


In [27]:
# Test 2
result2 = app.invoke({"messages": [HumanMessage(content="What is the square root of 256?")]})
print(result2["messages"][-1].content)

The square root of **256** is **16**! 🎉

This makes sense because 16 × 16 = 256. Let me know if you have any other math questions!


In [28]:
# Test 3 - multi tool
result3 = app.invoke({"messages": [HumanMessage(content="What is 5 to the power of 4, then find the square root of that?")]})
print(result3["messages"][-1].content)

Here's the full breakdown:
1. **5⁴ = 625**
2. **√625 = 25**

The final answer is **25**! 🎉 This makes sense mathematically because taking the square root of a number raised to the 4th power is the same as squaring it: 5² = 25.
